# Supervised CoNLL NER training

This notebook trains the final real or synthetic CoNLL model families. Run `step_1.ipynb` first, then materialize the datasets from the repository root:

```bash
python tools/materialize_datasets.py
```

All generated checkpoints are written to the ignored `artifacts/models/` directory. Change `USE_SYNTHETIC_DATA` and rerun the two code cells to train the corresponding family.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.supervised_training_conll import SupervisedTrainerConll

## Configuration

Set `USE_SYNTHETIC_DATA` to choose the corpus. The notebook trains each of the four entity-specific models and the combined model for all three encoder bases.

In [ ]:
USE_SYNTHETIC_DATA = False
NER_EPOCHS = 10
DROP_EMPTY_SENTENCE_PROB = 0.5
SKIP_IF_OUTPUT_EXISTS = False

DATA_DIR = PROJECT_ROOT / "data"
SPLITS_DIR = DATA_DIR / "splits"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "models"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

TAG_COMBINATIONS = [
    ["REACT"],
    ["ACTIVE"],
    ["FREQ"],
    ["SYS"],
    ["REACT", "ACTIVE", "FREQ", "SYS"],
]

if USE_SYNTHETIC_DATA:
    TRAIN_CSV = SPLITS_DIR / "train_synth.csv"
    TEST_CSV = SPLITS_DIR / "test_synth.csv"
    FULL_ANNOTATIONS_CSV = SPLITS_DIR / "full_annotations_synth.csv"
    TXT_DIR = DATA_DIR / "medicamentos_synth" / "txt"
    DATASET_PREFIX = "SYNTH-"
else:
    TRAIN_CSV = SPLITS_DIR / "train.csv"
    TEST_CSV = SPLITS_DIR / "test.csv"
    FULL_ANNOTATIONS_CSV = SPLITS_DIR / "full_annotations.csv"
    TXT_DIR = DATA_DIR / "medicamentos" / "txt"
    DATASET_PREFIX = ""

MODELS = [
    {
        "name": "CIMA-BERTIN-4.8",
        "pretrained_path": ARTIFACTS_DIR / "CIMA-BERTIN-4.8",
        "output_prefix": "CIMA-BERTIN-4.8-NER-ADR-CONLL",
    },
    {
        "name": "RoBERTa-biomedical-baseline",
        "pretrained_path": "PlanTL-GOB-ES/roberta-base-biomedical-es",
        "output_prefix": "RoBERTa-baseline-NER-ADR-CONLL",
    },
    {
        "name": "CIMA-RoBERTa-4.8",
        "pretrained_path": ARTIFACTS_DIR / "CIMA-RoBERTa-4.8",
        "output_prefix": "CIMA-RoBERTa-4.8-NER-ADR-CONLL",
    },
]

print(f"Dataset: {'synthetic' if USE_SYNTHETIC_DATA else 'real'}")
print(f"Training runs: {len(TAG_COMBINATIONS) * len(MODELS)}")

## Train all configurations

The next cell runs the Cartesian product of `TAG_COMBINATIONS` and `MODELS`. Set `SKIP_IF_OUTPUT_EXISTS` when resuming an interrupted batch.

In [ ]:
def _output_suffix(tags):
    return DATASET_PREFIX + "-".join(tag.lower() for tag in tags)


def train_one(model_config, tags):
    output_dir = ARTIFACTS_DIR / (
        f"{model_config['output_prefix']}-{_output_suffix(tags)}"
    )
    if SKIP_IF_OUTPUT_EXISTS and output_dir.exists():
        print(f"Skipping existing output: {output_dir}")
        return None

    print(f"Training {model_config['name']} with tags={tags}")
    trainer = SupervisedTrainerConll(
        pretrained_model_path=str(model_config["pretrained_path"]),
        output_dir=str(output_dir),
        ner_entity_types=tags,
        txt_dir=str(TXT_DIR),
    )

    train_df = trainer.load_csv_data(str(TRAIN_CSV))
    test_df = trainer.load_csv_data(str(TEST_CSV))
    full_df = trainer.load_csv_data(str(FULL_ANNOTATIONS_CSV))
    if train_df is None or test_df is None or full_df is None:
        raise FileNotFoundError("Training splits are incomplete; materialize the datasets first.")

    trainer.import_split_data(train_df, test_df, full_annotations=full_df)
    trainer.prepare_ner_conll_data()
    trainer.tokenize_and_align_ner_labels(
        drop_empty_sentence_prob=DROP_EMPTY_SENTENCE_PROB,
        drop_seed=42,
    )
    trainer.train_ner_conll_model(num_train_epochs=NER_EPOCHS)
    trainer.save_ner_label_mapping()
    return output_dir


completed_runs = []
for tags in TAG_COMBINATIONS:
    for model_config in MODELS:
        output_dir = train_one(model_config, tags)
        if output_dir is not None:
            completed_runs.append(output_dir)

print(f"Completed {len(completed_runs)} training runs.")